
# 大语言模型中的分词（Tokenization）核心笔记

## 1. 什么是分词？为什么需要分词？
在大模型（LLM）中，分词是将人类可读的连续文本切分为机器可处理的基本单元（**词元，Token**）的过程。

### 为什么当前主流采用**子词级（Subword）**分词？
* **字符级（Character-level）**：词表极小，但序列极长，导致模型处理效率低，且丢失了词汇层面的语义信息。
* **单词级（Word-level）**：语义保留完整，但会导致词表极其庞大（数百万），带来巨大的 Embedding 参数开销，且无法解决**未登录词（OOV, Out of Vocabulary）**问题。
* **子词级（Subword-level）**：在字符和单词之间取得平衡。将生僻词拆解为常见的子单元，既压缩了序列长度，缓解了 OOV 问题，又有效控制了词表规模。

---

## 2. 核心子词分词算法对比

### 2.1 字节对编码 (Byte-Pair Encoding, BPE) * **核心思想**：基于**频率**。从基础字符集开始，不断迭代寻找语料库中出现频率最高的相邻符号对，并将它们合并为新符号。
* **字节级 BPE (Byte-level BPE, BBPE)**：现代 LLM（如 GPT 系列、LLaMA）的主流选择。它不以 Unicode 字符为基础，而是以 **256 个 UTF-8 字节**为基础。
    * *优势*：彻底消除 OOV 问题，任何极其罕见的字符（如生僻汉字、特殊 Emoji）即使没见过，也可以回退到由基础字节拼接而成。
* **特点**：无损可逆，压缩效率高，保留了常见的前缀和后缀（如 `un-`, `-ing`），有助于模型泛化。

### 2.2 WordPiece
* **核心思想**：基于**最大化似然估计（互信息）**。与 BPE 类似都是自底向上的合并，但 WordPiece 不单纯看频率，而是看合并两个子词后，能否最大程度地提升语言模型对整体语料的概率（或者说两个子词组合在一起的互信息最高）。
* **应用**：由 Google 提出，最著名的应用是 **BERT**。在输出时，非词首的子词通常会带上 `##` 前缀（例如 `jet` 和 `##setter`）。

### 2.3 Unigram
* **核心思想**：基于**概率的剪裁**。与 BPE 自底向上的合并相反，Unigram 是**自顶向下**的。它先假设一个包含所有可能子词的超大候选词表，然后基于语言模型损失函数，逐步**剔除**那些对整体语料概率贡献最小的子词，直到达到预设的词表大小。
* **特点**：它是一种概率化分词，同一个单词可以有多种切分方式。在解码时，通常使用 Viterbi 算法选取概率最大的一条切分路径。它天然支持**子词正则化（Subword Regularization）**，即在训练时随机采样不同的切分方式，以增强模型的鲁棒性。

### 2.4 SentencePiece 工具
* **定位**：它不是一种单一算法，而是一个**分词工具包**（由 Google 开源）。
* **核心优势**：将**空格（Space）直接视为普通字符**（通常用 `_` 或 Unicode `U+2581` 表示）。这意味着它可以直接对原始文本进行训练和切分，不需要像早期英文分词那样依赖繁琐的规则预处理（去空格、标点切分）。对中文、日文等不以空格分词的语言尤其友好。
* **应用**：内部通常封装了 BPE 和 Unigram 算法，被广泛应用于 T5, ALBERT, LLaMA 等模型中。

---

## 3. 不同语言的分词策略与挑战

### 3.1 英文等基于空格的语言
传统做法是先按空格切分单词，再做子词切分。但现代采用 SentencePiece 或 BBPE 后，空格常作为 Token 的一部分被保留（例如 GPT-2 中的 `Ġhello`，`Ġ` 代表前置空格）。

### 3.2 中文等无空格语言的痛点
中文由于没有明显的词边界，纯数据驱动的子词算法如果直接在全量文本上训练，极易产生**跨词边界的错误合并**。
* **语义割裂或噪声**：例如“他是学科技的”，算法可能会因为“科技”和“的事”（如果语料中常出现）频率高，错误地切分并合并出与当前语境不符的怪异词元。
* **退化问题**：如 Google 的 `bert-base-chinese`，因为没有使用优秀的中文预分词策略，直接对单字进行 WordPiece，导致模型实际上退化成了“字符级（单字级）分词”，词表中塞满了无用的 `##` 汉字组合，浪费了参数。

### 3.3 优秀多语言模型的应对
* **扩充词表**：在原版英文 LLM 基础上（如 LLaMA），加入大量中文语料重新训练 Tokenizer，扩充几万个中文词汇（如词语级 Token：“人工智能”、“机器学习”），以提高对中文的编码效率。
* **强制预切分**：许多针对中文优化的模型，在训练 BPE 之前，会先使用正则表达式或结巴分词等工具进行一次粗粒度的预切分，强行划定安全边界，防止算法跨界合并。

---

## 4. 分词对模型核心能力的影响

1.  **训练与推理效率 (Efficiency)**：
    * **词表过大**：模型头尾的 Embedding 层参数量暴增，占用过多显存，且部分罕见 Token 训练不充分。
    * **词表过小（切分过细）**：同一个句子被切成太多 Token，序列长度飙升。由于 Transformer 的注意力机制计算复杂度是序列长度的平方，这会极大拖慢训练和推理速度。
2.  **语义理解准确性 (Semantics)**：
    * 优秀的子词切分能保留构词法规律（如词根、词缀），帮助模型举一反三理解新词（如德语的超长复合词）。
    * 糟糕的切分会破坏词语的完整语义，引入不必要的噪音，直接拉低模型在下游任务的表现。

---

## 💡 原资料中的瑕疵与补充说明

你提供的图片资料整体质量很高，但在一些技术细节的表述上不够严谨，容易让初学者产生误解。我为你梳理了以下几点修正：

1.  **关于 GPT-2 / 现代 BPE 的“空格预切分”描述不够准确**
    * *图片表述*：“GPT系列分词器采用了字节级 BPE，不严格依赖空格预切分。在这种机制下，空格会作为普通符号参与合并...” (图6)
    * *修正*：这句话极易产生误导。虽然 BBPE *理论上*可以不依赖预切分，但**实际上 GPT-2、GPT-3 甚至 LLaMA 都采用了极其严格的正则表达式（Regex）进行预分词（Pre-tokenization）**。例如，GPT-2 会使用正则强制将字母、数字、标点符号、空格隔离开来，防止 BPE 把单词和紧挨着的标点（比如 `dog.`）合并成一个词。图片忽略了预分词这一极其关键的清洗步骤。
2.  **关于 BPE 示例中的 `</w>` 结束符**
    * *图片表述*：在图3的详细示例中，每个单词末尾加上了 `</w>` 作为结束符进行合并。
    * *修正*：这种带 `</w>` 的做法是早期机器翻译领域（如 Subword NMT 工具）的经典 BPE 做法。而在现代 LLM（如基于 HuggingFace Tokenizers 或 SentencePiece 的模型）中，很少再显式地向语料中插入 `</w>`。现代的做法是像图 6 提到的那样，将**单词前的空格**作为区分标记（如 `Ġ` 或 `_`），或者依靠预切分划定边界。虽然原理图没错，但和现代 LLM 的实际工程实现有代差。
3.  **对“多语言词表扩大”的效率代价未做充分警示**
    * *图片表述*：图 7 提到 XLM-R 使用了高达 250,000 的词元。
    * *补充*：词表并非越大越好。词表从 5 万扩大到 25 万，虽然缩短了多语言的序列长度，但会导致模型的 Embedding 参数量膨胀数倍。在模型总参数不变的情况下，这意味着分配给 Transformer 核心计算层（注意力层和前馈层）的参数被挤压，可能会导致模型的深层推理能力下降。这是一个非常残酷的**算力 Trade-off**，资料中强调了对序列长度的压缩，但对词表膨胀的代价提及较弱。


In [1]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [3]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from transformers import AutoTokenizer

# 我们选取三个极具代表性的模型进行对比
model_names = [
    "bert-base-chinese",       # 代表：早期的 WordPiece（在中文上退化为纯单字切分）
    "gpt2",                    # 代表：传统的纯英文 BPE（对中文极不友好，会把汉字切成碎字节）
    "Qwen/Qwen2-0.5B"          # 代表：现代优秀的开源多语言大模型（扩充了中文词表，按词语切分）
]

# 测试文本（引用自你的学习资料）
text = "他是学科技的，他不相信神秘和超自然的事物"

print(f"【测试原句】: {text}")
print("=" * 60)

for model_name in model_names:
    try:
        # 加载对应的分词器
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # 将文本切分为 tokens (字符串形式)
        tokens = tokenizer.tokenize(text)
        # 将文本编码为 token IDs (数字形式)
        token_ids = tokenizer.encode(text, add_special_tokens=False)
        
        print(f"🤖 模型: {model_name}")
        print(f"📊 Token 数量: {len(tokens)}")
        print(f"✂️  切分结果: {tokens}")
        decoded_tokens = [tokenizer.decode([t]) for t in tokenizer.convert_tokens_to_ids(tokens)]
        print(f"📖 还原显示: {decoded_tokens}")
        print("-" * 60)
        
    except Exception as e:
        print(f"加载模型 {model_name} 的 Tokenizer 时出错: {e}")

【测试原句】: 他是学科技的，他不相信神秘和超自然的事物
🤖 模型: bert-base-chinese
📊 Token 数量: 20
✂️  切分结果: ['他', '是', '学', '科', '技', '的', '，', '他', '不', '相', '信', '神', '秘', '和', '超', '自', '然', '的', '事', '物']
📖 还原显示: ['他', '是', '学', '科', '技', '的', '，', '他', '不', '相', '信', '神', '秘', '和', '超', '自', '然', '的', '事', '物']
------------------------------------------------------------
🤖 模型: gpt2
📊 Token 数量: 41
✂️  切分结果: ['ä»', 'ĸ', 'æĺ¯', 'åŃ', '¦', 'ç', '§', 'ĳ', 'æ', 'Ĭ', 'Ģ', 'çļĦ', 'ï', '¼', 'Į', 'ä»', 'ĸ', 'ä¸į', 'çĽ', '¸', 'ä¿', '¡', 'ç¥ŀ', 'ç', '§', 'ĺ', 'å', 'Ĵ', 'Į', 'è', '¶ħ', 'è', 'ĩ', 'ª', 'çĦ', '¶', 'çļĦ', 'äº', 'ĭ', 'çī', '©']
📖 还原显示: ['�', '�', '是', '�', '�', '�', '�', '�', '�', '�', '�', '的', '�', '�', '�', '�', '�', '不', '�', '�', '�', '�', '神', '�', '�', '�', '�', '�', '�', '�', '��', '�', '�', '�', '�', '�', '的', '�', '�', '�', '�']
------------------------------------------------------------
🤖 模型: Qwen/Qwen2-0.5B
📊 Token 数量: 13
✂️  切分结果: ['ä»ĸæĺ¯', 'åŃ¦', 'ç§ĳæĬĢ', 'çļĦ', 'ï¼Į', 'ä»ĸ', 'ä¸įçĽ¸ä¿¡', 'ç¥ŀç